# People Clustering

Semantic search lets us find photos by *content*. The "People" album feature asks a different question: across thousands of photos, which faces belong to the same person? We do not know the identities in advance (there is no pre-defined label set), so this is an unsupervised problem. The pipeline has three stages: **detect** faces in each photo, **embed** each detected face into a metric space where same-person faces cluster, and **cluster** those embeddings into person identities.

This notebook covers all three stages, plus the identity management layer that maps clusters to `Person` database records.

## Face Detection

**Detection vs. recognition.** Face *detection* finds bounding boxes: it answers "where are the faces in this image?" Face *recognition* (or embedding) answers "who is this person?" We need both: detection first, to locate and crop faces, then embedding to produce a representation suitable for clustering.

We use **MTCNN** (Multi-task Cascaded Convolutional Networks) for detection. MTCNN is a three-stage cascade:

1. **P-Net** (Proposal Network): a fully-convolutional sliding window that produces a dense map of face proposals at multiple scales. Cheap and fast; high recall, many false positives.
2. **R-Net** (Refine Network): filters the P-Net proposals with a more powerful CNN; reduces false positives substantially.
3. **O-Net** (Output Network): refines bounding boxes to sub-pixel accuracy and regresses five **facial landmarks**: eyes, nose, and mouth corners.

The `facenet-pytorch` package implements MTCNN with pretrained weights. `MTCNN(keep_all=True)` returns all detected faces rather than just the highest-confidence one.

Defining the `FaceDetection` dataclass and the `detect_faces` function:

In [ ]:
from dataclasses import dataclass
from PIL import Image
import numpy as np

@dataclass
class FaceDetection:
    bbox      : tuple[float, float, float, float]  # (x1, y1, x2, y2) in pixels
    confidence: float
    landmarks : np.ndarray                          # shape (5, 2): [x, y] per keypoint


def detect_faces(image: Image.Image) -> list[FaceDetection]:
    """Detect all faces in a PIL image and return bounding boxes + landmarks."""
    from facenet_pytorch import MTCNN
    detector = MTCNN(keep_all=True, device="cpu")

    boxes, probs, landmarks = detector.detect(image, landmarks=True)
    if boxes is None:
        return []

    results = []
    for box, prob, lm in zip(boxes, probs, landmarks):
        results.append(FaceDetection(
            bbox       = tuple(float(v) for v in box),
            confidence = float(prob),
            landmarks  = np.array(lm, dtype=np.float32),
        ))
    return results

Testing the detector on a synthetic image with a solid-color rectangle as a placeholder face:

In [ ]:
import numpy as np
from PIL import Image

# Synthetic 256x256 image: gray background, skin-tone rectangle in the center
arr = np.full((256, 256, 3), 200, dtype=np.uint8)  # gray background
arr[80:176, 96:160] = [224, 172, 105]              # skin-tone rectangle
synthetic_img = Image.fromarray(arr)

detections = detect_faces(synthetic_img)
print(f"Detections found: {len(detections)}")
for d in detections:
    print(f"  bbox={d.bbox}, confidence={d.confidence:.3f}")

**Remark.** MTCNN was trained on real face images; a solid-color rectangle will not trigger a detection. In production, test with real portrait photographs. The synthetic image here exercises the detection pipeline end-to-end without requiring a photograph asset.

## Face Embedding

**FaceNet** embeds a cropped, aligned face image into $\mathbb{R}^{128}$. It is trained with **triplet loss**: for each anchor face $\mathbf{a}$, a positive (same person) $\mathbf{p}$, and a negative (different person) $\mathbf{n}$, the loss encourages

$$\|f(\mathbf{a}) - f(\mathbf{p})\|_2^2 + \alpha \leq \|f(\mathbf{a}) - f(\mathbf{n})\|_2^2$$

where $\alpha$ is a margin hyperparameter. After training, same-person embeddings cluster tightly in $\mathbb{R}^{128}$ and different-person embeddings are pushed apart by at least $\alpha$. The `facenet_pytorch.InceptionResnetV1(pretrained="vggface2")` variant was fine-tuned on the VGGFace2 dataset of 3.3M images across 9,131 identities.

**Alignment** is essential: the MTCNN landmarks are used to warp each face crop into a canonical pose (eyes at fixed pixel positions) before embedding. `facenet-pytorch` performs this alignment automatically when you call `mtcnn(image)` in crop mode.

Defining `embed_face` using InceptionResnetV1:

In [ ]:
import torch
from torchvision import transforms

# Load FaceNet once at module level
_face_transform = transforms.Compose([
    transforms.Resize((160, 160)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])


def embed_face(face_crop: Image.Image) -> np.ndarray:
    """Embed a cropped face image, returning a normalized (128,) float32 array."""
    from facenet_pytorch import InceptionResnetV1

    resnet = InceptionResnetV1(pretrained="vggface2").eval()
    tensor = _face_transform(face_crop).unsqueeze(0)  # (1, 3, 160, 160)
    with torch.no_grad():
        vec = resnet(tensor)                           # (1, 512) — project to 128 via norm
        vec = vec / vec.norm(dim=-1, keepdim=True)
    return vec.squeeze(0).numpy().astype(np.float32)

Verifying the output shape and norm:

In [ ]:
# Synthetic 160x160 face crop (uniform gray)
fake_crop = Image.fromarray(np.full((160, 160, 3), 128, dtype=np.uint8))
vec = embed_face(fake_crop)
print(f"Embedding shape : {vec.shape}")
print(f"L2 norm         : {np.linalg.norm(vec):.6f}")

**Integration.** For each ingested photo we (1) detect all faces with MTCNN, (2) embed each crop with FaceNet, and (3) insert a row into the `faces` table:

```python
INSERT INTO faces (photo_id, bbox, embedding, confidence)
VALUES (:photo_id, :bbox, :embedding, :confidence)
```

The `faces.embedding` column is `vector(512)` (the raw InceptionResnetV1 output dimension before any projection).

## Clustering Theory

**Why clustering, not classification.** We do not know the set of people in advance. Every time a new person appears in the photo library, they become a new class. Traditional classifiers require a fixed class set and a labelled training set, neither of which is available here. Clustering discovers groups purely from the geometry of the embedding space.

<br>

**DBSCAN** groups points that are within distance $\epsilon$ of at least `min_samples` other points into a *core point*, expanding the cluster through density-reachability. Points that are reachable from a core but are not themselves core points become *border points*. Points unreachable from any core are labelled $-1$ (noise). Two parameters: $\epsilon$ and `min_samples`. DBSCAN finds arbitrarily shaped clusters but requires careful tuning of $\epsilon$ — a single global threshold is a poor fit for face embeddings, where some people have thousands of photos and others appear only once.

<br>

**HDBSCAN** (Hierarchical DBSCAN) constructs a hierarchy of all possible DBSCAN clusterings across all values of $\epsilon$ simultaneously, then extracts a flat clustering by finding the most stable clusters in the hierarchy. The key advantages for face clustering:

- Single parameter `min_cluster_size` (minimum number of faces to form a person cluster).
- Handles variable-density clusters: a celebrity with 500 photos and a colleague with 5 photos can coexist in the same run.
- Noise points ($-1$) are handled gracefully: low-quality or partial faces are left unclustered.

<br>

**The curse of dimensionality.** In high dimensions, Euclidean distance concentrates — all pairwise distances become nearly equal, which destroys the cluster structure. Cosine distance is more robust in $\mathbb{R}^{512}$ because it measures angular separation, which remains discriminative even as $d$ grows. We pass `metric="cosine"` to HDBSCAN.

:::{.callout-note}
HDBSCAN with `metric="cosine"` requires that `hdbscan` is installed (`uv add hdbscan`). As of version 0.8.x the package also requires `scikit-learn >= 0.22`.

:::

Generating synthetic face embedding data and running HDBSCAN:

In [ ]:
import hdbscan
from sklearn.decomposition import PCA

rng = np.random.default_rng(7)

N_PEOPLE   = 5
SAMPLES    = 20    # faces per person
NOISE_PTS  = 10    # outlier detections
D          = 128   # embedding dimension

# Each person is a cluster centered at a random unit vector
centers = rng.standard_normal((N_PEOPLE, D))
centers /= np.linalg.norm(centers, axis=1, keepdims=True)

embeddings = np.vstack([
    centers[i] + rng.standard_normal((SAMPLES, D)) * 0.08
    for i in range(N_PEOPLE)
] + [rng.standard_normal((NOISE_PTS, D))])
embeddings /= np.linalg.norm(embeddings, axis=1, keepdims=True)  # normalize

true_labels = np.array(
    [i for i in range(N_PEOPLE) for _ in range(SAMPLES)] + [-1] * NOISE_PTS
)

# Run HDBSCAN
clusterer = hdbscan.HDBSCAN(min_cluster_size=5, metric="cosine")
pred_labels = clusterer.fit_predict(embeddings)

n_clusters = len(set(pred_labels)) - (1 if -1 in pred_labels else 0)
n_noise    = (pred_labels == -1).sum()
print(f"Found {n_clusters} clusters (expected {N_PEOPLE})")
print(f"Noise points: {n_noise}")

Visualizing the clusters in 2-D via PCA:

In [ ]:
#| code-fold: true
%config InlineBackend.figure_formats = ['svg']
import matplotlib.pyplot as plt

pca = PCA(n_components=2, random_state=0)
coords = pca.fit_transform(embeddings)

palette = plt.cm.tab10.colors
colors  = [
    palette[lbl % 10] if lbl >= 0 else (0.6, 0.6, 0.6)
    for lbl in pred_labels
]

fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(coords[:, 0], coords[:, 1], c=colors, s=18, alpha=0.85, linewidths=0)
ax.set_xlabel("PC 1"); ax.set_ylabel("PC 2")
ax.set_title("HDBSCAN clusters (PCA projection)")
ax.grid(linestyle="dotted", alpha=0.5)
plt.tight_layout();

**Figure.** PCA projection of the synthetic face embeddings. Each color represents a predicted cluster (person); gray points are HDBSCAN noise ($-1$ label). The PCA projection is lossy — clusters that appear to overlap in 2-D are typically well-separated in $\mathbb{R}^{128}$.

## Identity Management

Cluster labels are integers assigned by HDBSCAN; we need to translate them into stable `Person` database records. The mapping is: one unique non-$(-1)$ cluster label $\rightarrow$ one `Person` row. Faces with label $-1$ are left with `person_id = NULL`.

**Incremental clustering.** When new photos arrive, we embed their faces, append those embeddings to the existing matrix, and re-run HDBSCAN on the full set, with no need to retrain a model. Because HDBSCAN is deterministic for a fixed input order (with a fixed random seed), re-running it with the same embeddings in the same order returns the same labels — making the assignment **idempotent** as long as we use upsert semantics: insert a `Person` if the cluster ID has not been seen before, otherwise skip.

**Merge / split.** Errors are inevitable: the same person may be split into two clusters (slightly different lighting produces two tight groups), or two people may be merged into one (siblings who look alike). We support:

- **Merge**: user selects two `Person` records → we update all `faces.person_id` from the secondary cluster to the primary, then delete the secondary `Person` row.
- **Split**: user selects a `Person` record and marks a subset of their faces as belonging to a new person → we create a new `Person` row and reassign the selected face rows.

Implementing `assign_identities` and verifying idempotency:

In [ ]:
from dataclasses import dataclass, field

@dataclass
class PersonAssignment:
    face_index: int
    cluster_id: int          # -1 means noise / unknown
    person_id : int | None   # maps cluster_id -> a stable Person PK


def assign_identities(
    face_embeddings: np.ndarray,
    labels: np.ndarray,
    existing_cluster_to_person: dict[int, int] | None = None,
) -> tuple[list[PersonAssignment], dict[int, int]]:
    """
    Map HDBSCAN cluster labels to stable person IDs.

    Returns (assignments, updated_cluster_map) where cluster_map is suitable
    for passing back on the next incremental run (idempotency).
    """
    cluster_to_person: dict[int, int] = dict(existing_cluster_to_person or {})
    next_person_id = max(cluster_to_person.values(), default=0) + 1

    assignments: list[PersonAssignment] = []
    for i, lbl in enumerate(labels):
        lbl = int(lbl)
        if lbl == -1:
            assignments.append(PersonAssignment(face_index=i, cluster_id=-1, person_id=None))
            continue
        if lbl not in cluster_to_person:
            cluster_to_person[lbl] = next_person_id  # <1>
            next_person_id += 1
        assignments.append(PersonAssignment(
            face_index=i,
            cluster_id=lbl,
            person_id=cluster_to_person[lbl],
        ))

    return assignments, cluster_to_person


# --- First run ---
assignments_1, cluster_map = assign_identities(embeddings, pred_labels)

# --- Second run (idempotency check) ---
assignments_2, _           = assign_identities(embeddings, pred_labels, cluster_map)

ids_1 = [a.person_id for a in assignments_1]
ids_2 = [a.person_id for a in assignments_2]
print(f"Idempotent: {ids_1 == ids_2}")
print(f"Person IDs assigned: {sorted(set(i for i in ids_1 if i is not None))}")
print(f"Noise faces: {sum(1 for a in assignments_1 if a.person_id is None)}")

1. A new cluster label seen for the first time gets a fresh `person_id`. On subsequent runs the same label maps to the same `person_id`, ensuring upsert semantics.

## Album Generation

With `faces.person_id` populated we can expose three endpoints in the People router:

- `GET /people/`: list all persons with face count and representative photo (highest-confidence face).
- `GET /people/{person_id}/photos`: all photos containing this person; joins `faces` to `photos`.
- `POST /people/{person_id}/name`: set a human-readable name for a cluster (e.g., "Mom").

The representative photo query selects the face row with the maximum confidence for each `person_id` and joins to the photo metadata.

Defining the People router:

In [ ]:
PEOPLE_ROUTER = '''
from fastapi import APIRouter, Depends, HTTPException
from sqlalchemy.ext.asyncio import AsyncSession
from sqlalchemy import select, func
from pydantic import BaseModel

from .database import get_session
from .models   import Face, Photo, Person

router = APIRouter(prefix="/people", tags=["people"])


class PersonRead(BaseModel):
    id             : int
    name           : str | None
    face_count     : int
    cover_s3_key   : str | None

    model_config = {"from_attributes": True}


class NameRequest(BaseModel):
    name: str


@router.get("/", response_model=list[PersonRead])
async def list_people(session: AsyncSession = Depends(get_session)):
    # Subquery: best face (highest confidence) per person
    best_face = (
        select(
            Face.person_id,
            func.max(Face.confidence).label("max_conf"),
        )
        .where(Face.person_id.isnot(None))
        .group_by(Face.person_id)
        .subquery()
    )
    stmt = (
        select(
            Person.id,
            Person.name,
            func.count(Face.id).label("face_count"),
            Photo.s3_key.label("cover_s3_key"),
        )
        .join(Face, Face.person_id == Person.id)
        .join(best_face, (best_face.c.person_id == Person.id) & (Face.confidence == best_face.c.max_conf))
        .join(Photo, Photo.id == Face.photo_id)
        .group_by(Person.id, Photo.s3_key)
    )
    rows = (await session.execute(stmt)).mappings().all()
    return [PersonRead(**r) for r in rows]


@router.get("/{person_id}/photos", response_model=list[dict])
async def person_photos(
    person_id: int,
    session: AsyncSession = Depends(get_session),
):
    stmt = (
        select(Photo)
        .join(Face, Face.photo_id == Photo.id)
        .where(Face.person_id == person_id)
        .distinct(Photo.id)
    )
    photos = (await session.scalars(stmt)).all()
    return [{"id": p.id, "s3_key": p.s3_key, "taken_at": str(p.taken_at)} for p in photos]


@router.post("/{person_id}/name", response_model=PersonRead)
async def set_person_name(
    person_id: int,
    body: NameRequest,
    session: AsyncSession = Depends(get_session),
):
    person = await session.get(Person, person_id)
    if person is None:
        raise HTTPException(status_code=404, detail="Person not found")
    person.name = body.name
    await session.commit()
    await session.refresh(person)
    return person
'''
print(PEOPLE_ROUTER.strip())

## Appendix: Quality Filtering {#sec-quality-filter}

Not every detected face is worth embedding. Low-quality detections degrade cluster quality — an out-of-focus face crop or a tiny distant face will produce an unreliable embedding that adds noise. We apply three filters before calling `embed_face`:

1. **Confidence threshold.** `confidence < 0.90` → skip. MTCNN's output probability is a reliable proxy for detection quality.
2. **Blur filter.** Compute the variance of the Laplacian of the grayscale face crop. A high variance means sharp edges are present; a low variance indicates blur. `var < threshold` → skip.
3. **Minimum face size.** A face crop smaller than `min_pixels` area is unlikely to carry enough detail for a reliable embedding. `(x2 - x1) * (y2 - y1) < min_pixels` → skip.

Implementing the quality filter:

In [ ]:
import cv2

def laplacian_variance(image: Image.Image) -> float:
    """Higher = sharper. Used as a blur score."""
    gray = np.array(image.convert("L"), dtype=np.float32)
    return float(cv2.Laplacian(gray, cv2.CV_64F).var())


def is_usable_face(
    detection: FaceDetection,
    image: Image.Image,
    min_confidence: float = 0.90,
    min_blur_var  : float = 50.0,
    min_pixels    : float = 1600.0,  # 40x40 px minimum
) -> bool:
    """Return True iff the face detection meets all quality thresholds."""
    x1, y1, x2, y2 = detection.bbox

    if detection.confidence < min_confidence:        # <1>
        return False

    area = (x2 - x1) * (y2 - y1)
    if area < min_pixels:                            # <2>
        return False

    crop = image.crop((int(x1), int(y1), int(x2), int(y2)))
    if laplacian_variance(crop) < min_blur_var:      # <3>
        return False

    return True


# --- Demonstrate on a synthetic detection ---
fake_detection = FaceDetection(
    bbox=(10.0, 10.0, 90.0, 90.0),
    confidence=0.95,
    landmarks=np.zeros((5, 2), dtype=np.float32),
)
test_image = Image.fromarray(np.random.randint(0, 255, (128, 128, 3), dtype=np.uint8))
print(f"Usable (sharp noise image): {is_usable_face(fake_detection, test_image)}")

blurry = Image.fromarray(np.full((128, 128, 3), 128, dtype=np.uint8))
print(f"Usable (flat/blurry image): {is_usable_face(fake_detection, blurry)}")

1. Rejects detections below the MTCNN confidence threshold, which typically corresponds to partial or heavily occluded faces.
2. Rejects faces that are too small to contain useful detail; $40 \times 40$ pixels is a common lower bound in the literature.
3. Crops the bounding box from the full image before computing the Laplacian variance: we want to measure sharpness of the face region, not the background.

:::{.callout-caution}
The `min_blur_var` threshold is image-resolution-dependent. A value of `50.0` works well for face crops from typical smartphone photos (12–50 MP). Test and calibrate this threshold on your actual dataset before deploying.

:::

---

■